In [2]:
import tomllib
from pathlib import Path
import numpy as np
import pandas as pd
import os
from gensim.models import Word2Vec
from annoy import AnnoyIndex

import scipy
import json
import gensim
from nltk import sent_tokenize
import nltk
nltk.download('punkt')
nltk.download('stopwords')

from nltk.corpus import stopwords
stopWords = set(stopwords.words('english'))
stopwords = nltk.corpus.stopwords.words('english')

import h5py
import tqdm

import itertools
import gensim, logging
from gensim.corpora import Dictionary
from gensim.models.tfidfmodel import TfidfModel
from gensim.matutils import sparse2full

logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)
import gc
from multiprocessing import Pool
# from gensim.test.utils import datapath
from gensim.models.word2vec import Text8Corpus
from gensim.models.phrases import Phrases, ENGLISH_CONNECTOR_WORDS
from gensim.parsing.preprocessing import remove_stopwords


ModuleNotFoundError: No module named 'annoy'

In [ ]:
with open("./config.toml", "rb") as f:
    config = tomllib.load(f)
    
INPUT_FILE = Path(config["path"]["input"]) / "input\CN_abstracts.parquet"
OUTPUT_FOLDER = Path(config["path"]["output"])
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# hyps
min_count = config["bigram"]["min_count"]
threshold = config["bigram"]["threshold"]
vector_size = config["Word2vec"]["vector_size"]
window = config["Word2vec"]["window"]
min_count = config["Word2vec"]["min_count"]
trees = config["annoy"]["trees"]
search_k = config["similarity"]["search_k"]
threshold = config["similarity"]["threshold"]
upto = config["similarity"]["upto"]

WORKERS = int(max(os.cpu_count()/2, 1))

In [ ]:
# Preprocessing (sentence tokenizer + basik Gensim - lowercases, tokenizes, de-accents (optional))
def prepro(x):
    x = sent_tokenize(x)
    clean = []
    for j in x:
        clean.append(gensim.utils.simple_preprocess(j, min_len=1)) # min_len=1 -> keeps single letter token
    return clean
    
def bigrammer(x):
    sents = []
    for i in x:
        sents.append(bigram[i])
    return sents
    
def tfidfer(para):
    return model_tfidf[docs_dict.doc2bow(itertools.chain(*para))] 

def preprocess_1 (data):
    data['appln_abstract_prepro'] = list(map(prepro, data.appln_abstract_st_r))
    return data
    
def preprocess_2 (data):
    data['appln_abstract_prepro_bi'] = list(map(bigrammer, data['appln_abstract_prepro']))
    return data

def preprocess_3 (data):
    data['tfidf'] = list(map(tfidfer, data['appln_abstract_prepro_bi']))
    return data

In [ ]:
# data = pd.read_csv('https://raw.githubusercontent.com/AI-Growth-Lab/Patent_p2p_similarity_w2v/main/patent_dtatset_sample_1k.csv')
# data = data[['appln_id', 'appln_abstract']]

In [3]:
data = pd.read_parquet(INPUT_FILE, engine="pyarrow")

NameError: name 'INPUT_FILE' is not defined

In [9]:
# remove stop words from abstracts
tokens_without_sw = [remove_stopwords(text) for text in data.appln_abstract]
data['appln_abstract_st_r'] = tokens_without_sw

In [ ]:
data = preprocess_1(data) # add prepro

In [ ]:
# build bigrams
sentences = [sent for doc in data['appln_abstract_prepro'] for sent in doc if sent]
bigram = Phrases(sentences, min_count=min_count, threshold=threshold, connector_words=ENGLISH_CONNECTOR_WORDS)
bigram = gensim.models.phrases.Phraser(bigram)
bigram.save('bigram.m')

In [ ]:
data = preprocess_2(data) # add bigram

In [40]:
vector_size=300
window=80
min_count=1
WORKERS=14

In [ ]:
# word to vec
sentences = [sent for doc in data['appln_abstract_prepro_bi'] for sent in doc if sent]
model_with_phrases = Word2Vec(sentences=sentences, vector_size=vector_size, window=window,
                              min_count=min_count, workers=WORKERS)
model_with_phrases.save('w2v.m')

In [ ]:
# Build Gensim Dictionary
voc = model_with_phrases.wv.index_to_key
docs_dict = Dictionary([voc])
docs_dict.compactify()
docs_dict.save('docs_dict_11_2.d')

In [ ]:
# tfidf at patent level
docs = [list(itertools.chain(*doc)) for doc in data['appln_abstract_prepro_bi']]
model_tfidf = TfidfModel((docs_dict.doc2bow(d) for d in docs), id2word=docs_dict)
model_tfidf.save('model_tfidf_11_2.m')

docs_vecs = (sparse2full(model_tfidf[docs_dict.doc2bow(d)], len(docs_dict)) for d in docs)

In [ ]:
data = preprocess_3(data)
data.to_parquet("data.parquet", engine="pyarrow", index=False)

In [ ]:
# Selected Word-Embeddings
emb_vecs_selftrained = np.vstack([model_with_phrases.wv[docs_dict.get(i)] for i in range(len(docs_dict))])
n_abstracts = len(data)
appln_ids = data['appln_id'].to_numpy()          # same order as docs / docs_vecs

with h5py.File('docvecs_23_4_test.h5', 'w') as h5f:     # 'w' = overwrite, safe to rerun
    dataset = h5f.create_dataset('weighted_tfidf_docvecs', (n_abstracts, 300), dtype='f4')
    h5f.create_dataset('appln_id', data=appln_ids)

    a = np.vstack([next(docs_vecs) for _ in range(n_abstracts)])
    dataset[:] = np.dot(a, emb_vecs_selftrained)


In [ ]:
h5f = h5py.File('docvecs_23_4_test.h5', 'r')
dataset = h5f['weighted_tfidf_docvecs']
ids = h5f['appln_id'][:] # get every element
row_of = {aid: i for i, aid in enumerate(ids)}      # appln_id -> row

def vec(appln_id):
    return dataset[row_of[appln_id]]

scipy.spatial.distance.cosine(vec(ids[9]), vec(ids[8]))   # or vec(<a real appln_id>)


In [ ]:
# build trees
t = AnnoyIndex(dataset.shape[1])
t.on_disk_build('vecs_build.annoy')
for row in tqdm.tqdm(range(len(dataset))):
    t.add_item(row, dataset[row])       # Annoy item = row number
t.build(trees)                          # paper: 100
t.unload()                              # release the file (needed on Windows before re-loading)

In [ ]:
tt = AnnoyIndex(300)
tt.load('vecs_build.annoy', prefault=False)

simH5 = h5py.File('similarities_23_2.h5', 'a')
dataset = simH5.create_dataset('sims_50',(10000000,3), maxshape=(None,3), compression="gzip")

In [ ]:
pbar = tqdm.tqdm(total=tt.get_n_items())

blocks = []
start = 0
end = 0
counter = 0
for i in range(tt.get_n_items()):
    j = 100
    tresh = 0.0
    while tresh < threshold:
        y = tt.get_nns_by_item(i, j, search_k=search_k, include_distances=True)
        tresh = y[1][-1]
        j += 50
        if j >= upto:
            break
    block = np.array(list(zip([i for _ in range(len(y[0]))],y[0],y[1])))
    end = end + len(block)
    blocks.append(block)
    counter += 1
    if counter == 1000:
        counter = 0
        blocks_np = np.vstack(blocks)
        if end >= dataset.shape[0]:
            dataset.resize((end+(end - dataset.shape[0]),3))
        dataset[start:end] = blocks_np
        blocks = []
        gc.collect()
        start = end
        pbar.update(1000)
        
simH5.close()

In [ ]:
def lookup(x):
    y = tt.get_nns_by_item(x, j, search_k=5, include_distances=True)
    return np.array(list(zip([x for _ in range(len(y[0]))],y[0],y[1])))

In [ ]:
lookup(0)
    